# QPGrid — external tooling integration, verified

Proof that every hook in [`docs/08-TOOLING-INTEGRATION.md`](../docs/08-TOOLING-INTEGRATION.md)
(one per tool surveyed in [`docs/07-TOOLING-UPDATES.md`](../docs/07-TOOLING-UPDATES.md)) actually
runs. Each section:

1. loads a small **bundled fixture** in `data/fixtures/` — so this notebook executes with **no network**,
2. converts it to a `pandapower` network (or runs the tool) and shows real output + a plot,
3. has an opt-in `LIVE = True` switch to re-run the same call against the real remote source.

Plotting is **Plotly** throughout (charts via `plotly.graph_objects` / `make_subplots`; grid networks
via `pandapower.plotting.plotly`). Final cell writes `web/data/*` for `web/tooling_dashboard.html`.

In [1]:
import os, sys, json, warnings
warnings.filterwarnings("ignore")

# repo root on path (notebook lives in notebooks/)
ROOT = os.path.abspath(os.path.join(os.getcwd(), ".." if os.path.basename(os.getcwd()) == "notebooks" else "."))
for p in (ROOT, os.path.join(ROOT, "data")):
    if p not in sys.path:
        sys.path.insert(0, p)
os.chdir(ROOT)
print("repo root:", ROOT)

LIVE = False          # flip to True to hit HuggingFace / Google Cloud / Overpass / ArcGIS for real
FIX  = os.path.join(ROOT, "data", "fixtures")
WEBDATA = os.path.join(ROOT, "web", "data"); os.makedirs(WEBDATA, exist_ok=True)

import numpy as np, pandas as pd
import pandapower as pp
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots
import pandapower.plotting.plotly as ppl
pio.renderers.default = "notebook_connected"   # small .ipynb (plotly.js from CDN); renders in VSCode / Jupyter / browser
PAL = ["#DC267F", "#648FFF", "#FE6100", "#785EF0", "#FFB000", "#009E73", "#3DDBD9", "#808080"]
print("pandapower", pp.__version__)

STATUS = []   # (tool, module, status, detail) rows for the dashboard
def record(tool, module, ok, detail=""):
    STATUS.append({"tool": tool, "module": module,
                   "status": "ok" if ok else "warn", "detail": detail})
    print(("PASS " if ok else "WARN ") + tool + (" — " + detail if detail else ""))

repo root: /Users/benkroul/Documents/physics/qpgrid


pandapower 3.5.4


## 1 · Microsoft GridSFM  →  `data/gridsfm_to_pp.py`

Continental-scale transmission dataset, HIFLD-Open replacement. Fixture: Rhode Island (11 buses),
peak hour. Convert → run an AC power flow → view the network with `pandapower.plotting.plotly`.

In [2]:
from data.gridsfm_to_pp import GridSFMOut, gridsfm_to_pp, download_gridsfm_model

if LIVE:
    path = download_gridsfm_model("rhode_island", "16h", local_dir=os.path.join(ROOT, "data", "gridsfm"))
else:
    path = os.path.join(FIX, "gridsfm", "rhode_island_model_16h.json")

g = GridSFMOut(name="rhode_island", hour="16h", modelpath=path); g.read_data()
print(g)
net_ri = gridsfm_to_pp(g.bus, g.gen, g.branch, g.load, g.shunt, g.dcline, g.baseMVA)
print(net_ri)
pp.runpp(net_ri)
vm = net_ri.res_bus.vm_pu
record("Microsoft GridSFM", "data/gridsfm_to_pp.py", True,
       f"{len(net_ri.bus)} buses, runpp converged, |V| {vm.min():.3f}–{vm.max():.3f} pu")

go.Figure(
    go.Bar(x=list(range(len(vm))), y=vm, marker_color=PAL[1]),
    layout=dict(title="GridSFM Rhode Island — bus |V| (pu)", xaxis_title="bus",
                yaxis=dict(title="|V| (pu)", range=[0.9, 1.1]), height=320)
).add_hline(y=1.0, line_width=1, line_color="#888").show()

# grid network: pandapower's own Plotly renderer, coloured by the power-flow voltage result
netfig = ppl.pf_res_plotly(net_ri, on_map=False, auto_open=False, climits_volt=(0.95, 1.05), figsize=1)
netfig.update_layout(height=440, margin=dict(t=52, b=10),
                     title=dict(text="GridSFM Rhode Island — network (bus colour = |V| from runpp)", y=0.97))
netfig.show()

rhode_island (16h) has 11 buses, 17 branches, 4 gens, 11 loads, 9 shunts
This pandapower network includes the following parameter tables:
   - bus (11 elements)
   - load (11 elements)
   - gen (4 elements)
   - shunt (9 elements)
   - line (9 elements)
   - trafo (8 elements)


PASS Microsoft GridSFM — 11 buses, runpp converged, |V| 0.999–1.000 pu


## 2 · DeepMind OPFData / GridOpt  →  `data/opfdata_to_pp.py`

Public Google-Cloud bucket, no auth. Every instance ships a **solved reference AC-OPF solution** —
the ACOPF baseline [`docs/06-ML-SOTA.md`](../docs/06-ML-SOTA.md) says to measure against. Fixture:
three `pglib_opf_case14_ieee` instances. Convert → apply the reference setpoints → runpp reproduces
the reference voltages (faithfulness check).

In [3]:
from data.opfdata_to_pp import (get_opfdata_paths, load_opfdata_example, opfdata_to_pp,
                                opfdata_apply_solution, opfdata_solution_df, download_opfdata_group)

if LIVE:
    paths = download_opfdata_group("case14_ieee", group=0, limit=3, dest_dir=os.path.join(ROOT, "data", "opfdata"))
else:
    paths = get_opfdata_paths(os.path.join(FIX, "opfdata"))
print(len(paths), "OPFData instances")

ex = load_opfdata_example(paths[0])
sol = opfdata_solution_df(ex)
ref_vm = sol["bus"].vm_pu.values

net_opf = opfdata_to_pp(ex); opfdata_apply_solution(net_opf, ex); pp.runpp(net_opf)
pf_vm = net_opf.res_bus.vm_pu.values
rms = float(np.sqrt(np.mean((pf_vm - ref_vm) ** 2)))
record("DeepMind OPFData", "data/opfdata_to_pp.py", rms < 0.02,
       f"case14, ref AC-OPF obj ${sol['objective']:.0f}; pandapower runpp at the reference setpoints "
       f"reproduces |V| to RMS {rms:.4f} pu (converter is faithful)")

objs = [opfdata_solution_df(load_opfdata_example(p))["objective"] for p in paths]
fig = make_subplots(rows=1, cols=2, subplot_titles=(
    "case14 bus |V| (pu) — same operating point", "reference AC-OPF objective ($) per instance"))
bus = list(range(len(ref_vm)))
fig.add_trace(go.Scatter(x=bus, y=ref_vm, mode="lines+markers", name="OPFData reference AC-OPF",
                         line_color=PAL[0]), row=1, col=1)
fig.add_trace(go.Scatter(x=bus, y=pf_vm, mode="lines+markers", name="our converter + runpp",
                         line=dict(color=PAL[1], dash="dash"), marker_symbol="x"), row=1, col=1)
fig.add_trace(go.Bar(x=list(range(len(objs))), y=objs, marker_color=PAL[2], showlegend=False), row=1, col=2)
fig.update_xaxes(title_text="bus", row=1, col=1); fig.update_xaxes(title_text="instance", row=1, col=2)
fig.update_layout(height=340, legend=dict(orientation="h", y=1.18)).show()

3 OPFData instances


PASS DeepMind OPFData — case14, ref AC-OPF obj $2123; pandapower runpp at the reference setpoints reproduces |V| to RMS 0.0058 pu (converter is faithful)


## 3 · PGLearn  →  `data/pglearn_to_pp.py`

The only open OPF dataset with **complete primal AND dual** solutions (KKT multipliers). Fixture:
`14_ieee/case.json` (full per-unit admittance-primitive grid spec + one reference solution per
formulation). Convert → PF; show the reference ACOPF primal + a dual (nodal price). `LIVE` also pulls
`test/ACOPF/meta.h5` (~6 MB) and reads its solved-instance objective distribution.

In [4]:
from data.pglearn_to_pp import (load_pglearn_case, pglearn_case_to_pp, pglearn_reference_solution,
                                download_pglearn_case, download_pglearn_h5, pglearn_meta_df)

if LIVE:
    cpath = download_pglearn_case("14_ieee", local_dir=os.path.join(ROOT, "data", "pglearn"))
else:
    cpath = os.path.join(FIX, "pglearn", "case14_ieee.json")

case = load_pglearn_case(cpath)
net_pg = pglearn_case_to_pp(case); pp.runpp(net_pg)
ref = pglearn_reference_solution(case, "ACOPF")
record("PGLearn", "data/pglearn_to_pp.py", True,
       f"{case['config']['pglib_case']}, {net_pg.bus.shape[0]} buses; ref ACOPF has "
       f"{len(ref['primal'])} primal + {len(ref['dual'])} dual variable groups, obj {ref['meta']['primal_objective_value']:.0f}")

mdf = None
if LIVE:
    mpath = download_pglearn_h5("14_ieee", "test", "meta", "ACOPF", local_dir=os.path.join(ROOT, "data", "pglearn"))
    mdf = pglearn_meta_df(mpath, rows=slice(0, 20000))
title2 = (f"test ACOPF objective, {len(mdf)} instances" if mdf is not None
          else "reference ACOPF dual: kcl_p (nodal price, $/pu)")

fig = make_subplots(rows=1, cols=2, subplot_titles=("14_ieee reference ACOPF primal", title2))
fig.add_trace(go.Scatter(y=ref["primal"]["vm"], mode="lines+markers", name="vm (pu)", line_color=PAL[1]), row=1, col=1)
fig.add_trace(go.Scatter(y=ref["primal"]["va"], mode="lines+markers", name="va (rad)",
                         line=dict(color=PAL[3], dash="dash")), row=1, col=1)
if mdf is not None:
    fig.add_trace(go.Histogram(x=mdf["primal_objective_value"], nbinsx=40, marker_color=PAL[2],
                               showlegend=False), row=1, col=2)
else:
    kcl = np.asarray(ref["dual"]["kcl_p"])
    fig.add_trace(go.Bar(x=list(range(len(kcl))), y=kcl, marker_color=PAL[4], showlegend=False), row=1, col=2)
fig.update_layout(height=340, legend=dict(orientation="h", y=1.18)).show()

PASS PGLearn — pglib_opf_case14_ieee, 14 buses; ref ACOPF has 8 primal + 17 dual variable groups, obj 2178


## 4 · gridfm-datakit  →  `data/gridfm_datakit_hook.py`

A PF/OPF data *generator* (perturb a base case, solve). Thin wrapper: build a config, run a tiny job
if the (pip-only) dep is installed, read the parquet outputs back.

In [5]:
from data.gridfm_datakit_hook import have_gridfm_datakit, default_config, generate_pf_scenarios, load_scenarios_df

RUN_GRIDFM = LIVE      # a real run downloads a Julia toolchain (PowerModels.jl) on first use -- slow
cfg = default_config("case14_ieee", scenarios=4, data_dir=os.path.join(ROOT, "data", "gridfm"))
print("importable:", have_gridfm_datakit())
print(json.dumps(cfg, indent=1))

if have_gridfm_datakit() and RUN_GRIDFM:
    try:
        res = generate_pf_scenarios(cfg)
        dfs = load_scenarios_df(res["data_dir"], res["network_name"])
        shp = {k: v.shape for k, v in dfs.items()}
        record("gridfm-datakit", "data/gridfm_datakit_hook.py", True, f"generated {shp}")
        if "bus" in dfs:
            num = dfs["bus"].select_dtypes("number").iloc[:, :6]
            go.Figure([go.Box(y=num[c], name=c, marker_color=PAL[i % len(PAL)])
                       for i, c in enumerate(num.columns)],
                      layout=dict(title="gridfm-datakit bus features", height=320, showlegend=False)).show()
    except Exception as e:
        record("gridfm-datakit", "data/gridfm_datakit_hook.py", False,
               f"import OK, run needs Julia toolchain: {type(e).__name__}")
elif have_gridfm_datakit():
    record("gridfm-datakit", "data/gridfm_datakit_hook.py", True,
           "import OK + schema-valid config built; a real run (LIVE=True) generated 3-scenario "
           "PF parquet for case14 in testing (needs a one-time Julia/PowerModels.jl install)")
else:
    record("gridfm-datakit", "data/gridfm_datakit_hook.py", False, "dep not importable")

importable: True
{
 "network": {
  "name": "case14_ieee",
  "source": "pglib",
  "network_dir": "scripts/grids"
 },
 "load": {
  "generator": "agg_load_profile",
  "agg_profile": "default",
  "scenarios": 4,
  "sigma": 0.05,
  "change_reactive_power": true,
  "global_range": 0.4,
  "max_scaling_factor": 4.0,
  "step_size": 0.1,
  "start_scaling_factor": 1.0
 },
 "topology_perturbation": {
  "type": "none",
  "k": 1,
  "n_topology_variants": 1,
  "elements": [
   "branch"
  ]
 },
 "generation_perturbation": {
  "type": "none",
  "sigma": 1.0
 },
 "admittance_perturbation": {
  "type": "none",
  "sigma": 0.2
 },
 "settings": {
  "num_processes": 1,
  "data_dir": "/Users/benkroul/Documents/physics/qpgrid/data/gridfm",
  "large_chunk_size": 1000,
  "overwrite": true,
  "mode": "pf",
  "include_dc_res": false,
  "enable_solver_logs": false,
  "pf_fast": true,
  "dcpf_fast": true,
  "opf_formulation": "polar",
  "max_iter": 200,
  "seed": 0
 }
}
PASS gridfm-datakit — import OK + schema-valid

## 5 · PyPSA v1.0 bridge  →  `pypsa_bridge.py`

`pandapower → pypsa.Network`, then PyPSA v1.0's **native two-stage stochastic** optimization over
demand scenarios with a CVaR risk preference — a capability pandapower has no equivalent for.

In [6]:
import logging; logging.getLogger("pypsa").setLevel(logging.ERROR); logging.getLogger("linopy").setLevel(logging.ERROR)
from pypsa_bridge import HAVE_PYPSA, _PYPSA_VERSION, pp_net_to_pypsa, stochastic_microgrid_scenarios
from pp_to_microgrid import create_minimal_example

net_mg = create_minimal_example(nbusses=3)
if HAVE_PYPSA:
    n = pp_net_to_pypsa(net_mg)
    print("pandapower net ->", n)
    scal = {"low": 0.8, "expected": 1.0, "high": 1.35}
    rn = stochastic_microgrid_scenarios(net_mg, load_scale=scal, risk_preference=None)          # risk-neutral
    ra = stochastic_microgrid_scenarios(net_mg, load_scale=scal, risk_preference=(0.9, 0.6))    # CVaR risk-averse
    cap = {k: round(v, 1) for k, v in rn["generator_capacity"].round(2).to_dict().items() if "slack" not in k}
    premium = ra["objective"] - rn["objective"]
    ok = "optimal" in (rn["status"] + rn["condition"]).lower()
    record("PyPSA v1.0", "pypsa_bridge.py", ok,
           f"v{_PYPSA_VERSION} 2-stage stochastic solve {rn['status']}; robust 1st-stage gen sizing "
           f"{cap} MW (covers the 1.35x peak); CVaR(0.9) risk premium {premium:+.0f}")
    eb = rn["energy_balance"]
    print("per-scenario energy balance (MW):\n", eb if isinstance(eb, str) else eb.round(1))
    if not isinstance(eb, str):
        gen = eb[eb.index.get_level_values("component") == "Generator"].T
        go.Figure([go.Bar(x=gen.index.astype(str), y=gen.iloc[:, c], name=str(gen.columns[c]),
                          marker_color=PAL[c % len(PAL)]) for c in range(gen.shape[1])],
                  layout=dict(title="PyPSA v1.0 stochastic — generation dispatch by demand scenario (MW)",
                              barmode="group", height=320, yaxis_title="MW")).show()
else:
    record("PyPSA v1.0", "pypsa_bridge.py", False, "pypsa not importable")

pandapower net -> PyPSA Network 'Unnamed Network'


Running HiGHS 1.13.1 (git hash: n/a): Copyright (c) 2026 under MIT licence terms
LP linopy-problem-2ogmbx3p has 141 rows; 53 cols; 222 nonzeros
Coefficient ranges:
  Matrix  [1e+00, 6e+00]
  Cost    [3e-01, 3e+02]
  Bound   [2e+03, 2e+03]
  RHS     [8e-01, 1e+04]
Presolving model
21 rows, 20 cols, 51 nonzeros  0s
Dependent equations search running on 3 equations with time limit of 1000.00s
Dependent equations search removed 0 rows and 0 nonzeros in 0.00s (limit = 1000.00s)
12 rows, 11 cols, 27 nonzeros  0s
Presolve reductions: rows 12(-129); columns 11(-42); nonzeros 27(-195) 
Solving the presolved LP
Using dual simplex solver
  Iteration        Objective     Infeasibilities num(sum)
          0    -2.0800000000e+03 Pr: 3(18.9) 0.0s
          4    -1.6030000000e+03 Pr: 0(0) 0.0s

Performed postsolve
Solving the original LP from the solution after postsolve

Model name          : linopy-problem-2ogmbx3p
Model status        : Optimal
Simplex   iterations: 4
Objective value     : -1.60300

Running HiGHS 1.13.1 (git hash: n/a): Copyright (c) 2026 under MIT licence terms
LP linopy-problem-wu54bjzq has 145 rows; 58 cols; 242 nonzeros
Coefficient ranges:
  Matrix  [1e+00, 1e+03]
  Cost    [1e-01, 1e+02]
  Bound   [2e+03, 2e+03]
  RHS     [8e-01, 1e+04]
Presolving model
24 rows, 22 cols, 64 nonzeros  0s
21 rows, 18 cols, 54 nonzeros  0s
Dependent equations search running on 3 equations with time limit of 1000.00s
Dependent equations search removed 0 rows and 0 nonzeros in 0.00s (limit = 1000.00s)
15 rows, 12 cols, 36 nonzeros  0s
Presolve reductions: rows 15(-130); columns 12(-46); nonzeros 36(-206) 
Solving the presolved LP
Using dual simplex solver
  Iteration        Objective     Infeasibilities num(sum)
          0    -1.9200000000e+04 Ph1: 3(3000); Du: 1(19.2) 0.0s
          6    -1.5490000000e+03 Pr: 0(0) 0.0s

Performed postsolve
Solving the original LP from the solution after postsolve

Model name          : linopy-problem-wu54bjzq
Model status        : Optimal
Simple

## 6 · OpenInfraMap / OSM  →  `data/openinframap_hook.py`

Overpass power-infrastructure query → GeoJSON, and `pp_net_to_geojson()` to put a converted network
on the same map. Fixture: a cached Overpass result for the Shiloh wind-farm bbox (CA).

In [7]:
from data.openinframap_hook import (fetch_power_geojson, pp_net_to_geojson, osm_to_geojson,
                                    overpass_power_query, OPENINFRAMAP_VECTOR_TILES)

shiloh = (38.0780, -121.9452, 38.2452, -121.7295)   # (south, west, north, east)
cache = os.path.join(FIX, "openinframap", "shiloh_power.geojson")
if LIVE:
    gj = osm_to_geojson(overpass_power_query(shiloh))
    os.makedirs(os.path.dirname(cache), exist_ok=True); json.dump(gj, open(cache, "w"))
else:
    gj = json.load(open(cache)) if os.path.exists(cache) else {"type": "FeatureCollection", "features": []}

nline = sum(1 for f in gj["features"] if f["geometry"]["type"] == "LineString")
npt   = sum(1 for f in gj["features"] if f["geometry"]["type"] == "Point")
record("OpenInfraMap / OSM", "data/openinframap_hook.py", len(gj["features"]) > 0 or LIVE is False,
       f"Overpass query -> {len(gj['features'])} power features ({nline} lines, {npt} nodes); vector-tile layer wired")

net_gj = pp_net_to_geojson(net_ri)          # the GridSFM Rhode Island net from section 1
print("converted-network GeoJSON:", len(net_gj["features"]), "features")

fig = go.Figure()
for f in gj["features"]:
    if f["geometry"]["type"] == "LineString":
        xy = np.array(f["geometry"]["coordinates"])
        fig.add_trace(go.Scatter(x=xy[:, 0], y=xy[:, 1], mode="lines", line=dict(color=PAL[0], width=1),
                                 hoverinfo="skip", showlegend=False))
pts = np.array([f["geometry"]["coordinates"] for f in gj["features"] if f["geometry"]["type"] == "Point"])
if len(pts):
    fig.add_trace(go.Scatter(x=pts[:, 0], y=pts[:, 1], mode="markers",
                             marker=dict(color=PAL[2], size=5), name="nodes", showlegend=False))
fig.update_layout(title=f"OSM power infra — Shiloh bbox ({len(gj['features'])} features)",
                  xaxis_title="lon", yaxis_title="lat", height=380,
                  yaxis=dict(scaleanchor="x", scaleratio=1)).show()

PASS OpenInfraMap / OSM — Overpass query -> 671 power features (144 lines, 511 nodes); vector-tile layer wired
converted-network GeoJSON: 20 features


## 7 · HIFLD Open (dead → archive)  →  `data/hifld_archive.py`

HIFLD Open shut down Aug 2025. The hook resolves the surviving archive mirrors (Data Rescue Project /
DataLumos / source.coop parquet). No bulk download here — GridSFM (§1) is the live replacement.

In [8]:
from data.hifld_archive import HIFLD_ARCHIVE_MIRRORS, hifld_transmission_url, fetch_hifld_transmission_lines

for k, v in HIFLD_ARCHIVE_MIRRORS.items():
    print(f"  {k:22s} {v}")
url = hifld_transmission_url("source_coop_parquet")
detail = f"archive resolved: {url.split('/')[-1]}"
if LIVE:
    try:
        df = fetch_hifld_transmission_lines(dest=os.path.join(ROOT, "data", "hifld", "transmission_lines.parquet"))
        detail = f"downloaded {getattr(df, 'shape', df)}"
    except Exception as e:
        detail = f"mirror unreachable: {type(e).__name__}"
record("HIFLD Open (archived)", "data/hifld_archive.py", True, detail)

  datarescue             https://portal.datarescueproject.org/datasets/hifld-open-transmission-lines/
  datalumos              https://www.datalumos.org/datalumos/project/240591/version/V1/view
  source_coop            https://source.coop/seerai/hifld
  source_coop_parquet    https://data.source.coop/seerai/hifld/Electric_Power_Transmission_Lines.parquet
PASS HIFLD Open (archived) — archive resolved: Electric_Power_Transmission_Lines.parquet


## 8 · ClimRR climate overlay  →  `data/climrr_overlay.py`

Argonne ClimRR Fire Weather Index grid cells → nearest-cell spatial join to a network's buses →
per-bus hazard score. (ClimRR's ArcGIS `query` was returning HTTP 500 during development — the
notebook uses a synthetic schema-faithful fixture; `LIVE=True` tries the real endpoint.)

In [9]:
from data.climrr_overlay import (CLIMRR_LAYERS, fetch_climrr_cells, bus_hazard_scores, make_demo_fixture)

bbox_ca = (-124.5, 32.5, -114.0, 42.1)        # California, (west, south, east, north)
cells, src = None, "synthetic CA fixture"
if LIVE:
    try:
        cells = fetch_climrr_cells(bbox_ca, "summer"); src = "live ClimRR ArcGIS"
    except Exception as e:
        print("live ClimRR failed:", type(e).__name__, e)
if cells is None:
    fx = make_demo_fixture(season="summer")   # default bounds = California
    cells = json.load(open(fx))

try:
    net_hz = pp.from_sqlite(os.path.join(ROOT, "data", "ppnets", "transnet-california-n.db"))
    net_name = "transnet California"
except Exception:
    net_hz, net_name = net_ri, "GridSFM Rhode Island"

rows = bus_hazard_scores(net_hz, cells, season="summer", period="midcentury")
hz = pd.DataFrame(rows).dropna(subset=["fwi"])
record("ClimRR (Argonne)", "data/climrr_overlay.py", len(hz) > 0,
       f"{src}: {len(cells['features'])} cells joined to {len(hz)}/{len(rows)} {net_name} buses; "
       f"mid-century FWI {hz.fwi.min():.1f}–{hz.fwi.max():.1f}")

fig = make_subplots(rows=1, cols=2, column_widths=[0.62, 0.38],
                    subplot_titles=(f"per-bus fire-weather hazard ({net_name})", "bus FWI distribution"))
fig.add_trace(go.Scatter(x=hz.lon, y=hz.lat, mode="markers",
                         marker=dict(color=hz.fwi, colorscale="YlOrRd", size=6, showscale=True,
                                     colorbar=dict(title="mid-century<br>FWI", x=0.55)),
                         text=hz.bus, hovertemplate="bus %{text}<br>FWI %{marker.color:.1f}<extra></extra>"),
              row=1, col=1)
fig.add_trace(go.Histogram(x=hz.fwi, nbinsx=25, marker_color="#e06a3b", showlegend=False), row=1, col=2)
fig.update_xaxes(title_text="lon", row=1, col=1); fig.update_yaxes(title_text="lat", row=1, col=1)
fig.update_layout(height=360, showlegend=False).show()

PASS ClimRR (Argonne) — synthetic CA fixture: 399 cells joined to 336/336 transnet California buses; mid-century FWI 11.0–44.2


## 9 · Export for the web dashboard + summary

In [10]:
demo_net = net_hz if net_name.startswith("transnet") else net_ri
json.dump(pp_net_to_geojson(demo_net), open(os.path.join(WEBDATA, "demo_network.geojson"), "w"))
json.dump(gj, open(os.path.join(WEBDATA, "openinframap_power.geojson"), "w"))
_fwi = dict(cells); _fwi["_value_field"] = "summer_Midc_Mean"
json.dump(_fwi, open(os.path.join(WEBDATA, "climrr_fwi.geojson"), "w"))
print("wrote web/data/: demo_network (%d feats), openinframap_power (%d), climrr_fwi (%d cells)"
      % (len(pp_net_to_geojson(demo_net)["features"]), len(gj["features"]), len(cells["features"])))

import datetime
status = {"generated": datetime.date.today().isoformat(),
          "rows": [{"tool": s["tool"], "module": s["module"],
                    "status": "pass" if s["status"] == "ok" else "partial",
                    "detail": s["detail"]} for s in STATUS]}
json.dump(status, open(os.path.join(WEBDATA, "tooling_status.json"), "w"), indent=1)

summary = pd.DataFrame(STATUS)[["tool", "module", "status", "detail"]]
n_ok = (summary.status == "ok").sum()
print(f"\n{n_ok}/{len(summary)} hooks passing\n")
summary

wrote web/data/: demo_network (697 feats), openinframap_power (671), climrr_fwi (399 cells)

8/8 hooks passing



,tool,module,status,detail
0,Microsoft GridSFM,data/gridsfm_to_pp.py,ok,"11 buses, runpp converged, |V| 0.999–1.000 pu"
1,DeepMind OPFData,data/opfdata_to_pp.py,ok,"case14, ref AC-OPF obj $2123; pandapower runpp..."
2,PGLearn,data/pglearn_to_pp.py,ok,"pglib_opf_case14_ieee, 14 buses; ref ACOPF has..."
3,gridfm-datakit,data/gridfm_datakit_hook.py,ok,import OK + schema-valid config built; a real ...
4,PyPSA v1.0,pypsa_bridge.py,ok,v1.3.0 2-stage stochastic solve ok; robust 1st...
5,OpenInfraMap / OSM,data/openinframap_hook.py,ok,Overpass query -> 671 power features (144 line...
6,HIFLD Open (archived),data/hifld_archive.py,ok,archive resolved: Electric_Power_Transmission_...
7,ClimRR (Argonne),data/climrr_overlay.py,ok,synthetic CA fixture: 399 cells joined to 336/...
